In [1]:
pip install openai supabase

     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------  81.9/82.2 kB 1.6 MB/s eta 0:00:01
     ---------------------------------------- 82.2/82.2 kB 1.6 MB/s eta 0:00:00
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/48.4 kB ? eta -:--:--
   ---------------------------------------- 48.4/48.4 kB ? eta 0:00:00
   --------------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
"""
SmartReg Monitor - Ingesta de Chunks en Supabase con Embeddings
================================================================
Sprint 2/3 - PBI 3.1: Generación de embeddings con text-embedding-3-small

Lee chunks_all.json, genera embeddings con Azure OpenAI y sube cada fila
a la tabla legal_chunks en Supabase (PostgreSQL + pgvector).

Autores: Alonso Gómez, Ethan Macías, Borja Núñez
Proyecto: SmartReg Monitor - Ricoh Edition
"""

import json
import os
import time
from pathlib import Path

from openai import AzureOpenAI
from supabase import create_client

# ──────────────────────────────────────────────
# CONFIGURACIÓN — Pon aquí vuestras credenciales
# ──────────────────────────────────────────────

# Azure OpenAI
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "TU_AZURE_OPENAI_API_KEY")              # La API key de Azure OpenAI
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://foundryresource-ricoh.openai.azure.com/")           # El endpoint de vuestro recurso
AZURE_OPENAI_API_VERSION = "2024-06-01"                                # Versión de la API
AZURE_EMBEDDING_DEPLOYMENT = "text-embedding-3-small"                  # Nombre del DEPLOYMENT (ver Azure AI Studio → Deployments)

# Supabase
SUPABASE_URL = "https://lfgnjqfbwwqvuzekbtmx.supabase.co"
SUPABASE_KEY = os.environ.get("SUPABASE_KEY", "TU_SUPABASE_KEY")

# Configuración general
CHUNKS_FILE = "../json/chunk/chunks_all.json"   # Ruta al archivo de chunks
TABLE_NAME = "legal_chunks"


# ──────────────────────────────────────────────
# INICIALIZACIÓN DE CLIENTES
# ──────────────────────────────────────────────
openai_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

supabase_client = create_client(SUPABASE_URL, SUPABASE_KEY)


def generate_embedding(text: str) -> list[float]:
    """Llama a Azure OpenAI para generar el vector de un texto."""
    response = openai_client.embeddings.create(
        model=AZURE_EMBEDDING_DEPLOYMENT,   # En Azure, aquí va el nombre del deployment
        input=text
    )
    return response.data[0].embedding


def load_chunks(filepath: str) -> list[dict]:
    """Carga el JSON de chunks."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def ingest_chunks():
    """Pipeline principal: carga chunks, genera embeddings y sube a Supabase."""

    # 1. Cargar chunks
    chunks = load_chunks(CHUNKS_FILE)
    total = len(chunks)
    print(f"Cargados {total} chunks desde {CHUNKS_FILE}\n")

    ok = 0
    errores = 0

    for i, chunk in enumerate(chunks, start=1):
        label = f"{chunk.get('documento', '?')} - {chunk.get('articulo', '?')}"
        parte = chunk.get("parte")
        if parte:
            label += f" (parte {parte}/{chunk['total_partes']})"

        try:
            # 2. Generar embedding
            texto_para_embedding = chunk.get("contenido_sin_header") or chunk["contenido"]
            embedding = generate_embedding(texto_para_embedding)

            # 3. Preparar la fila para Supabase
            row = {
                "chunk_id":             chunk["chunk_id"],
                "documento":            chunk.get("documento"),
                "referencia_legal":     chunk.get("referencia_legal"),
                "capitulo":             chunk.get("capitulo"),
                "articulo":             chunk.get("articulo"),
                "titulo_articulo":      chunk.get("titulo_articulo"),
                "contenido":            chunk.get("contenido"),
                "contenido_sin_header": chunk.get("contenido_sin_header"),
                "num_tokens_estimados": chunk.get("num_tokens_estimados"),
                "jurisdiccion":         chunk.get("jurisdiccion"),
                "embedding":            embedding,
            }

            # 4. Upsert en Supabase (usa chunk_id como clave)
            supabase_client.table(TABLE_NAME).upsert(row, on_conflict="chunk_id").execute()

            ok += 1
            print(f"[{i}/{total}] ✅ Artículo subido: {label}")

        except Exception as e:
            errores += 1
            print(f"[{i}/{total}] ❌ Error en: {label} — {e}")

        # Pequeña pausa para no saturar la API
        time.sleep(0.25)

    # 5. Resumen final
    print(f"\n{'='*50}")
    print(f"INGESTA COMPLETADA")
    print(f"{'='*50}")
    print(f"  Subidos correctamente: {ok}/{total}")
    print(f"  Errores:               {errores}/{total}")


if __name__ == "__main__":
    ingest_chunks()

Cargados 9 chunks desde ../json/chunk/chunks_all.json

[1/9] ✅ Artículo subido: RGPD - Artículo 1
[2/9] ✅ Artículo subido: RGPD - Artículo 2
[3/9] ✅ Artículo subido: RGPD - Artículo 3
[4/9] ✅ Artículo subido: RGPD - Artículo 4
[5/9] ✅ Artículo subido: AI Act - Considerandos / Preámbulo (parte 1/2)
[6/9] ✅ Artículo subido: AI Act - Considerandos / Preámbulo (parte 2/2)
[7/9] ✅ Artículo subido: AI Act - Artículo 1
[8/9] ✅ Artículo subido: AI Act - Artículo 2 (parte 1/2)
[9/9] ✅ Artículo subido: AI Act - Artículo 2 (parte 2/2)

INGESTA COMPLETADA
  Subidos correctamente: 9/9
  Errores:               0/9
